# End-to-end Machine Learning Project
**CS 540 – Principles of AI · Catholic Polytechnic University · Summer 2026**  
Instructor: Marcus Birkenkrahe  
Source: Géron, *Hands-on Machine Learning with scikit-learn and PyTorch*, Ch. 2

## Objectives

1. Look at the big picture
2. Get the data
3. Explore and visualize the data
4. Create the test set
5. Prepare data for learning
6. Select a model and train
7. Fine-tune a model
8. Present your solution
9. Launch production system
10. Monitor and maintain production system

See also: [Géron's project checklist](https://github.com/ageron/handson-mlp/blob/main/ml-project-checklist.md)

## Source

Géron's *Hands-on Machine Learning with scikit-learn and PyTorch*, Ch. 2.  
The text is very densely written and has many side alleys and tangents (~60 pages of content).

You can open his notebook (mostly code) in Colab and go through it yourself:  
[tinyurl.com/geron-2-ipynb](https://colab.research.google.com/github/ageron/handson-mlp/blob/main/02_end_to_end_machine_learning_project.ipynb)

## Target Data

**Why start with data?**

> Machine learning is learning from data to create rules baked into a machine model so that the model can make predictions on new data. Therefore, data — not models, coding languages, or infrastructure — takes the center role.

**What data are we using and why?**

> 1. **Origin**: 1990 California Housing Prices dataset from the US Census Bureau ([see Kaggle](https://www.kaggle.com/datasets/harrywang/housing?select=housing.csv)).
> 2. **Metrics**: Population, median income, median housing price etc. by district (block group).
> 3. **Why**: Data are realistic (though outdated) and verifiable. *(Pace & Barry, Statistics & Probability Letters, 1997)*

## Frame the Problem and Look at the Big Picture

1. Define the objective in business terms.
2. How will your solution be used?
3. What are the current solutions/workarounds (if any)?
4. How should you frame this problem (supervised/unsupervised, online/offline, etc.)?
5. How should performance be measured?
6. Is the performance measure aligned with the business objective?
7. What would be the minimum performance needed to reach the business objective?
8. What are comparable problems? Can you reuse experience or tools?
9. Is human expertise available?
10. How would you solve the problem manually?
11. List the assumptions you or others have made so far.
12. Verify assumptions if possible.

### Is There a Business Need for Machine Learning?

- **Model output** = prediction of a district's median housing price
- **Goal** = decide whether it's worth investing in a given area
- Output will be fed to another ML system as part of a **data science pipeline**:
  - Upstream = close to the source (data)
  - Downstream = close to the project client (insights)
- Currently, a team of experts estimates house prices based on complex rules — costly, time-consuming, error-prone.

### How Would You Frame This as an ML Problem?

- Supervised, unsupervised, or reinforcement learning?
- Classification, regression, or something else?
- Batch learning or online learning?

**Answer:**

> - Model can be trained with labeled examples: each row comes with the expected output (district's median house price) → **supervised ML**
> - Model predicts a single continuous numeric value → **multiple** [predictors], **univariate** [predicted] **regression**
> - Data are static, small (1.42 MB), not changing fast → **batch learning** (use all available data at once, offline, then launch)

### How Would You Measure Model Performance?

For regression problems we use **RMSE** (Root Mean Squared Error):

$$\text{RMSE}(\mathbf{X}, h) = \sqrt{\frac{1}{m}\sum_{i=1}^{m}\left(h(\mathbf{x}^{(i)}) - y^{(i)}\right)^2}$$

Where:
- $m$ — number of instances (rows/observations)
- $\mathbf{x}^{(i)}$ — feature vector for instance $i$ (all features except the label)
- $h(\mathbf{x}^{(i)})$ — model prediction for instance $i$
- $y^{(i)}$ — actual label for instance $i$
- $\mathbf{X}$ — matrix of all feature values (one row per instance)

**Why use RMSE?**

> RMSE is in the same units as the target variable, making it directly interpretable.

**What is common to all performance measures?**

> All performance measures are functions of the feature matrix **X** and the model (hypothesis $h$): they aggregate the difference between predictions $h(\mathbf{x}^{(i)})$ and actual labels $y^{(i)}$ into a single scalar. The choice between them (RMSE, MAE, etc.) depends on how you want to weight errors — RMSE penalizes outliers more heavily (quadratic), MAE treats all errors equally (linear absolute value).

**Does the geometric argument for Lasso vs. Ridge also apply to MAE vs. RMSE?**

> Yes — the same L1/L2 geometry applies. MAE uses absolute values (L1, diamond constraint region with corners on the axes), RMSE uses squared errors (L2, circular constraint region). L1 is more robust to outliers because large errors are not amplified by squaring. Lasso/MAE share the diamond geometry; Ridge/RMSE share the circle geometry.

## Get the Data

### Checklist

*Note: automate as much as possible so you can easily get fresh data.*

1. List the data you need and how much you need.
2. Find and document where you can get that data.
3. Check how much space it will take.
4. Check legal obligations, and get authorization if necessary.
5. Get access authorizations.
6. Create a workspace (with enough storage space).
7. Get the data.
8. Convert the data to a format you can easily manipulate (without changing the data itself).
9. Ensure sensitive information is deleted or protected (e.g., anonymized).
10. Check the size and type of data (time series, sample, geographical, etc.).
11. Sample a test set, put it aside, and never look at it (no data snooping!).

### Fetch the Data

The data are available as a compressed tarball. We load it directly from a URL using pandas.

In [ ]:
import pandas as pd

housing_full = pd.read_csv("https://tinyurl.com/housing-csv")

### Get a Quick Overview

In [ ]:
print(housing_full.head())

In [ ]:
print(housing_full.info())

**What do you see?**

> 1. 10 features: nine numeric, one object (`ocean_proximity`)
> 2. 20,640 rows
> 3. `total_bedrooms` has 207 missing values
> 4. `ocean_proximity` is a categorical text feature

In [ ]:
print(housing_full["ocean_proximity"].value_counts())

`<1H OCEAN` = less than one hour from ocean.

In [ ]:
print(housing_full.describe())

**What do we learn?**

> 1. Missing values (`NaN`) are ignored by `describe()`.
> 2. `std` measures **spread** — it's pretty tight compared to the `mean` (**centrality**).
> 3. Outliers aren't dominating (compare `mean` and median/50th percentile).
> 4. IQR (Inter-Quartile Range) = 75th − 25th percentile.

You can make the IQR visible with a boxplot (length of the box):

In [ ]:
import matplotlib.pyplot as plt

housing_full.boxplot(column="median_house_value", figsize=(4, 6))
plt.title("Median House Value by District")
plt.ylabel("USD")
plt.show()

**What do you notice about this distribution?**

> The top of the whiskers appears capped/clipped. Capped values are a strong pattern and you don't want the model to learn that.

**What is capping/clipping/winsorizing?**

> This is not *truncating* (removing values) but clipping them to a chosen threshold — e.g., a house value of \$1,000,000 still appears in the data but is recorded as \$500,000 (the cap).

### Make a Quick Plot

A histogram shows frequency (value counts) for each continuous numerical feature. The `hist()` method produces histograms for the whole dataset:

In [ ]:
plt.clf()
housing_full.hist(bins=50, figsize=(12, 8))
plt.tight_layout()
plt.show()

**What do you see?**

> 1. `median_house_value` and `housing_median_age` appear **clipped** at the top.
> 2. `median_income` appears clipped at both ends and is not in USD (5–15 equates to \$50,000–\$150,000).
> 3. `total_rooms`, `total_bedrooms`, `population`, `households`, `median_income`, and `median_house_value` are all **right-skewed**: a few very large outlier values pull the mean above the median.
> 4. The features are on wildly different **scales** — scaling matters before training.
> 5. `median_income` is closest to bell-shaped: a candidate for **stratified sampling**.
> 6. `latitude` and `longitude` are bimodal (two spikes) — two distinct subpopulations. `housing_median_age` is even trimodal.

**Which features would you trust to feed directly into linear regression, and which would you transform first?**

> 1. **Direct** (non-skewed): `longitude`, `latitude`, `housing_median_age`, `median_income`.
> 2. **Transform** (log): `total_rooms`, `total_bedrooms`, `population`, `households`.
> 3. **Problematic regardless**: `median_house_value` (capping disrupts the target — high prices won't be predicted).
> 
> *Why?* Linear regression assumes a linear relationship between each feature and the target. Skewed features compress variation at the high end, making it harder for the model to detect the relationship. A log transform spreads values more evenly.

## Create a Test Set

### Why?

We need to evaluate the model on data it has never seen, to avoid **data snooping bias** — looking at the test set biases your intuitions and model choices. The test set must be set aside *before* any exploration.

### Random Sampling

The simplest approach: pick 20% of instances at random.

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(
    housing_full, test_size=0.2, random_state=42)
print(len(train_set), len(test_set))

**Problem**: with a skewed dataset, random sampling may introduce **sampling bias** — the test set may not represent the full population, especially in the tails.

**Worse**: if the script is re-run with a different `random_state`, a different 20% is selected each time. Over multiple runs the model indirectly sees the *whole dataset* — defeating the purpose of a test set entirely.

**Solution**: fix `random_state` (we use 42 by convention), or use a stable hash of a row identifier so new instances always get the same train/test assignment regardless of dataset order or size.

### Stratified Sampling

**Key insight**: `median_income` is the most important predictor of housing price. We must ensure the test set represents its distribution faithfully.

**Step 1**: cut `median_income` into categories.

In [ ]:
import numpy as np

housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5])

Visualize the income categories to confirm the strata are well-populated before splitting:

In [ ]:
housing_full["income_cat"].value_counts().sort_index().plot(
    kind="bar", figsize=(8, 4), rot=0)
plt.xlabel("Income category")
plt.ylabel("Number of districts")
plt.title("Income category distribution")
plt.show()

**Step 2**: split *within* each category.

In [ ]:
strat_train, strat_test = train_test_split(
    housing_full, test_size=0.2,
    stratify=housing_full["income_cat"],
    random_state=42)

**Step 3**: verify proportions match the full dataset.

In [ ]:
# Full dataset proportions
print(housing_full["income_cat"].value_counts() / len(housing_full))

# Test set proportions
print(strat_test["income_cat"].value_counts() / len(strat_test))

**Step 4**: drop the helper column.

In [ ]:
for df in (strat_train, strat_test):
    df.drop("income_cat", axis=1, inplace=True)

**What do we learn?**

> 1. **Stratified sampling** preserves the income distribution of the full dataset in both train and test sets. Random sampling does not guarantee this — especially in the tails where rare high-income districts (categories 1 and 5) could be over- or under-represented.
> 2. The **bar chart** confirms the strata are well-populated before splitting: category 3 dominates (most districts have middle incomes), categories 1 and 5 are rare but not empty.
> 3. The **proportion comparison** after splitting verifies that the test set mirrors the full dataset's income distribution faithfully — this is what "representative" means empirically.